# SpendDNA — Your Wallet's Year-End Story

- **Name**: Parvathy M
- **Batch**: June
- **Date**: 4 July 2026

In [86]:
import pandas as pd
import numpy as np

pd.set_option('display.max_rows', 20)
FILENAME = 'rahul_transactions.csv'

df_raw = pd.read_csv(FILENAME)
print(f"Raw file loaded: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
df_raw.head()

Raw file loaded: 1328 rows, 8 columns


,Date,Time,Description,Type,Amount,Balance,Mode,Ref
0,2024-01-01,03:11,AMAZON SELLER SVCS,Debit,₹2462,678275.0,UPI,TXN190872
1,01-Jan-24,05:44,BHIM-BMTC,DR,50.00,681007.0,UPI,TXN143064
2,01-Jan-24,09:35,NEFT-TECHCRUSH LABS-SALARY MAY24,CR,₹84728,484728.0,NEFT,TXN246316
3,2024-01-01,14:07,UPI-AMAN-8934@OKAXIS,Debit,₹1828,-748745.0,UPI,TXN569226
4,01 Jan 2024,14:23,BHIM-BLINKIT,Debit,270.00,680737.0,UPI,TXN968962


## Feature 1 - The Transaction Parser

The raw file has 4 date formats, 3 amount formats, 2 spellings of the type field, 18 duplicate rows, and blank Mode values. This cell cleans everything into a single 'df', that can be used for every following features.



In [87]:
# Drop duplicate rows
df = df_raw.drop_duplicates().reset_index(drop=True)
duplicates_dropped = len(df_raw)-len(df)

# Parse 4 mixed date format explicitly
date_formats = ['%d/%m/%y', '%Y-%m-%d', '%d-%b-%y', '%d %b %Y']
parsed_date = pd.Series(pd.NaT, index=df.index, dtype='datetime64[ns]')
for fmt in date_formats:
  still_unparsed = parsed_date.isna()
  parsed_date.loc[still_unparsed] = pd.to_datetime(df.loc[still_unparsed, 'Date'], format=fmt, errors='coerce')
df['date'] = parsed_date

# Standardize type field
df['type_clean'] = (df['Type'].astype(str).str.strip().str.lower()
    .replace({'dr': 'debit', 'cr': 'credit'}))

# Clean amount
amount_clean = (
    df['Amount'].astype(str)
    .str.replace('₹', '', regex=False)
    .str.replace('Rs.', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)
df['amount'] = pd.to_numeric(amount_clean, errors='coerce')

# Treat blank mode values as missing
df['Mode'] = df['Mode'].replace('', np.nan)

# Anything still didn't get parsed gets dropped
unparseable_amounts = df['amount'].isna().sum()
unparseable_dates = df['date'].isna().sum()
df = df.dropna(subset=['date', 'amount']).reset_index(drop=True)

# Print output
print(f"Parsed {len(df)} transactions across 6 months.")
print(f"Dropped {duplicates_dropped} duplicates.")
print(f"{unparseable_amounts} unparseable amounts, {unparseable_dates} unparseable dates.")
print()
print(df[['Date', 'date', 'Amount', 'amount', 'Type', 'type_clean']].dtypes)

Parsed 1310 transactions across 6 months.
Dropped 18 duplicates.
0 unparseable amounts, 0 unparseable dates.

Date                  object
date          datetime64[ns]
Amount                object
amount               float64
Type                  object
type_clean            object
dtype: object


## Feature 2 - Vendor Extractor

From each messy Description string, the canonical vendor name is extracted. A vendor dictionary is built, mapping keyword patterns to canonical names.

In [88]:
# Unique raw descriptions
print(f"Unique raw descriptions: {df['Description'].nunique()}")

Unique raw descriptions: 283


In [89]:
# Canonical name's keywords
vendor_keywords = {
    'Instamart':['INSTAMART'],
    'Blinkit':['BLINKIT', 'GROFERS', 'KIRANAKART'],
    'Zepto':['ZEPTO'],
    'Swiggy':['SWIGGY', 'BUNDL'],
    'Zomato Dining':['ZOMATO-DINING'],
    'Dineout':['DINEOUT'],
    'Zomato':['ZOMATO'],
    'Starbucks':['STARBUCKS'],
    'Third Wave Coffee':['THIRDWAVE', 'THIRD WAVE', 'TWC'],
    'Cafe Coffee Day':['COFFEE DAY', 'CCD'],
    'Restaurant':['RESTAURANT', 'MEGHANA FOODS', 'TRUFFLES'],
    'Amazon Prime':['AMAZON PRIME', 'AMZN PRIME'],
    'Amazon':['AMAZON', 'AMZN'],
    'Flipkart':['FLIPKART', 'FKART'],
    'Myntra':['MYNTRA'],
    'Nykaa':['NYKAA', 'FSN E-COMMERCE'],
    'DMart':['DMART', 'AVENUE SUPERMARTS'],
    'BigBasket':['BIGBASKET', 'INNOVATIVE RETAIL'],
    'Ola Electric':['OLA ELECTRIC'],
    'Ola':['OLA CABS', 'OLA-PRIME', 'OLACABS', 'ANI TECHNOLOGIES'],
    'Uber':['UBER'],
    'Rapido':['RAPIDO', 'ROPPEN'],
    'BMTC':['BMTC', 'TUMMOC'],
    'BPCL':['BPCL'],
    'HP Petrol':['HP PETROL'],
    'Indian Oil':['INDIAN OIL', 'IOC'],
    'Netflix':['NETFLIX'],
    'Spotify':['SPOTIFY'],
    'Disney+ Hotstar':['HOTSTAR', 'STAR INDIA'],
    'BESCOM':['BESCOM', 'BANGALORE ELEC'],
    'BWSSB':['BWSSB'],
    'Airtel':['AIRTEL'],
    'Vodafone Idea':['VI POSTPAID', 'VODAFONE IDEA', 'VI-RECHARGE'],
    'JIO':['JIO'],
    'Zerodha':['ZERODHA'],
    'Groww':['GROWW', 'NEXTBILLION'],
    'BookMyShow':['BOOKMYSHOW', 'BMS MOVIE', 'BIGTREE'],
    'Salary':['SALARY'],
    'Rent':['RENT-LANDLORD'],
    'Cash Withdrawal':['ATM-WDL'],
    'P2P Transfer':['AMAN', 'ANKIT', 'PRIYA', 'NEHA', 'KARAN', 'SNEHA', 'VIKAS'],
}

def extract_vendor(description):
  desc_upper = str(description).upper()
  for vendor, keywords in vendor_keywords.items():
    for kw in keywords:
      if kw in desc_upper:
        return vendor
  return 'Uncategorised'

df['vendor_clean'] = df['Description'].apply(extract_vendor)
print(f"Canonical vendors found: {df['vendor_clean'].nunique()}")
print()
print(df['vendor_clean'].value_counts().head(10))

Canonical vendors found: 41

vendor_clean
Swiggy        176
Zomato        101
Amazon         77
Uber           71
Blinkit        68
Instamart      67
Ola            67
Restaurant     63
Zepto          58
Rapido         55
Name: count, dtype: int64


In [90]:
# Vendor cleanup audit
uncategorised = df[df['vendor_clean'] == 'Uncategorised']['Description'].unique()
print(f"Uncategorised descriptions: {len(uncategorised)}")
print(uncategorised)

Uncategorised descriptions: 0
[]


## Feature 3 - Category Tagger

Every vendor is mapped to one of 12 categories: Food Delivery, Quick Commerce, Ecommerce, Transport, Cafe, Restaurants, Subscriptions, Utilities, Groceries, Investments, Fuel, Entertainment, plus Personal Transfer and Cash Withdrawal.

In [91]:
category_map = {
    'Food Delivery':['Swiggy', 'Zomato'],
    'Quick Commerce':['Instamart', 'Blinkit', 'Zepto'],
    'E-commerce':['Amazon', 'Flipkart', 'Myntra', 'Nykaa'],
    'Transport':['Ola', 'Ola Electric', 'Uber', 'Rapido', 'BMTC'],
    'Cafe':['Starbucks', 'Third Wave Coffee', 'Cafe Coffee Day'],
    'Restaurants':['Restaurant', 'Dineout', 'Zomato Dining'],
    'Subscriptions':['Netflix', 'Spotify', 'Disney+ Hotstar', 'Amazon Prime'],
    'Utilities':['BESCOM', 'BWSSB', 'Airtel', 'Vodafone Idea', 'JIO'],
    'Groceries':['DMart', 'BigBasket'],
    'Investments':['Zerodha', 'Groww'],
    'Fuel':['BPCL', 'HP Petrol', 'Indian Oil'],
    'Entertainment':['BookMyShow'],
    'Personal Transfer':['P2P Transfer'],
    'Cash Withdrawal':['Cash Withdrawal'],
    'Rent':['Rent'],
    'Income':['Salary'],
}

vendor_to_category = {vendor: cat for cat, vendors in category_map.items() for vendor in vendors}
df['category'] = df['vendor_clean'].map(vendor_to_category)

print(f"Categories in use: {df['category'].nunique()}")
print(df['category'].value_counts())

Categories in use: 16
category
Food Delivery        277
Transport            250
Quick Commerce       193
E-commerce           163
Cafe                  99
Restaurants           93
Utilities             43
Groceries             41
Subscriptions         40
Fuel                  28
Investments           23
Personal Transfer     18
Cash Withdrawal       17
Entertainment         13
Income                 6
Rent                   6
Name: count, dtype: int64


## Feature 4 - Spending Overview

Computes and prints the headline financial stats: total credits, total debits, net savings, savings rate, top 5 categories by spend, top 5 vendors by spend, total transaction count.

In [92]:
total_credits = df.loc[df['type_clean'] == 'credit', 'amount'].sum()
total_debits  = df.loc[df['type_clean'] == 'debit', 'amount'].sum()
net_change    = total_credits - total_debits
savings_rate  = net_change / total_credits * 100

spend_df = df[(df['type_clean'] == 'debit') & (~df['category'].isin(['Personal Transfer', 'Cash Withdrawal']))].copy()
spend_total = spend_df['amount'].sum()

category_totals = spend_df.groupby('category')['amount'].sum().sort_values(ascending=False)
category_pct    = category_totals / spend_total * 100

vendor_totals = spend_df.groupby('vendor_clean')['amount'].sum().sort_values(ascending=False)
vendor_counts = spend_df.groupby('vendor_clean').size()

print(f"Total credits    : Rs. {total_credits:,.0f}")
print(f"Total debits     : Rs. {total_debits:,.0f}")
print(f"Net change       : Rs. {net_change:,.0f}")
print(f"Savings rate     : {savings_rate:.1f}%")
print(f"Transactions     : {len(df)}")
print(f"Unique vendors   : {df['vendor_clean'].nunique()}")
print()
print("Top 5 categories (% of spend):")
print(category_pct.head(5).round(1))
print()
print("Top 5 vendors (by amount):")
print(vendor_totals.head(5).round(0))

Total credits    : Rs. 509,774
Total debits     : Rs. 1,678,901
Net change       : Rs. -1,169,127
Savings rate     : -229.3%
Transactions     : 1310
Unique vendors   : 41

Top 5 categories (% of spend):
category
E-commerce       36.9
Investments      15.4
Restaurants       7.9
Food Delivery     7.4
Rent              6.7
Name: amount, dtype: float64

Top 5 vendors (by amount):
vendor_clean
Amazon        318612.0
Zerodha       210000.0
Flipkart      177510.0
Rent          108000.0
Restaurant     97912.0
Name: amount, dtype: float64


## Feature 5 - Monthly Trend Analysis

A category-by-month matrix (Jan→Jun), used to spot which category is trending up and which is winding
down. Growth is measured as % change from January to June for each category.

In [93]:
month_names = {1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun'}
spend_df['month'] = df.loc[spend_df.index, 'date'].dt.month

month_pivot = spend_df.pivot_table(values='amount', index='category', columns='month', aggfunc='sum', fill_value=0)
month_pivot = month_pivot.rename(columns=month_names)
print(month_pivot.round(0))

# Jan->Jun % growth per category (NumPy-style % change)
jan_col, jun_col = 1, 6
growth_pct = ((month_pivot['Jun'] - month_pivot['Jan']) / month_pivot['Jan'].replace(0, np.nan) * 100).sort_values(ascending=False)

print()
print(f"Fastest-growing category   : {growth_pct.index[0]}  ({growth_pct.iloc[0]:+.1f}% Jan->Jun)")
print(f"Fastest-declining category : {growth_pct.index[-1]}  ({growth_pct.iloc[-1]:+.1f}% Jan->Jun)")

month               Jan      Feb       Mar      Apr      May       Jun
category                                                              
Cafe             3690.0   4273.0    5448.0   6564.0   5668.0    5802.0
E-commerce      97134.0  92773.0  103772.0  68098.0  95191.0  136991.0
Entertainment    1263.0    474.0    2418.0   2224.0      0.0    1914.0
Food Delivery   20206.0  18714.0   19166.0  20726.0  21193.0   19496.0
Fuel            30322.0   2079.0   26164.0  18718.0   9138.0    2882.0
Groceries       17649.0   8571.0    6289.0  11748.0   9718.0    5432.0
Investments     38432.0  15000.0   68644.0  54126.0  48628.0   23330.0
Quick Commerce  12797.0  17465.0   17979.0  16572.0  15188.0   15666.0
Rent            18000.0  18000.0   18000.0  18000.0  18000.0   18000.0
Restaurants     17004.0  24510.0   29997.0  10039.0  23260.0   22480.0
Subscriptions    4256.0   5866.0    7952.0   3289.0   2429.0    4597.0
Transport       11005.0  10191.0    6857.0   9284.0  13141.0    6996.0
Utilit

## Feature 6 - Time-of-Day Patterns

We check what fraction of Food Delivery orders happen late at night (9 PM–2 AM) and what
fraction of Cafe spend happens in the morning commute window (8 AM–11 AM).

In [94]:
df['hour'] = df['Time'].str[:2].astype(int)

# Category * hour matrix
categories_list = sorted(spend_df['category'].dropna().unique())
cat_hour_matrix = np.zeros((len(categories_list), 24))
spend_df['hour'] = df.loc[spend_df.index, 'hour']

for i, cat in enumerate(categories_list):
    cat_hours = spend_df.loc[spend_df['category'] == cat, 'hour']
    for h in range(24):
        cat_hour_matrix[i, h] = (cat_hours == h).sum()

food_hours = spend_df.loc[spend_df['category'] == 'Food Delivery', 'hour']
late_night_food_pct = ((food_hours >= 21) | (food_hours < 2)).sum() / len(food_hours) * 100

cafe_hours = spend_df.loc[spend_df['category'] == 'Cafe', 'hour']
morning_cafe_pct = ((cafe_hours >= 8) & (cafe_hours <= 11)).sum() / len(cafe_hours) * 100

print(f"Food Delivery orders between 9 PM - 2 AM : {late_night_food_pct:.1f}%")
print(f"Cafe spend between 8 AM - 11 AM           : {morning_cafe_pct:.1f}%")
print()
print("Category x Hour matrix shape (rows=categories, cols=24 hours):", cat_hour_matrix.shape)

Food Delivery orders between 9 PM - 2 AM : 19.9%
Cafe spend between 8 AM - 11 AM           : 35.4%

Category x Hour matrix shape (rows=categories, cols=24 hours): (13, 24)


## Feature 7 - Anomaly Detection

Find transactions that are anomalously large for their category. For each category we compute the mean and standard deviation of transaction amounts within that category

In [95]:
category_mean = spend_df.groupby('category')['amount'].transform('mean')
category_std  = spend_df.groupby('category')['amount'].transform('std')
spend_df['z_score'] = (spend_df['amount'] - category_mean) / category_std

anomalies = spend_df[spend_df['z_score'] > 2].sort_values('z_score', ascending=False)

print(f"Anomalous transactions flagged (z > 2): {len(anomalies)}")
print()
anomalies_display = anomalies[['date', 'vendor_clean', 'category', 'amount', 'z_score']].head(5).copy()
anomalies_display['date'] = anomalies_display['date'].dt.strftime('%d %b')
anomalies_display['z_score'] = anomalies_display['z_score'].round(1)
print(anomalies_display.to_string(index=False))

Anomalous transactions flagged (z > 2): 24

  date vendor_clean    category  amount  z_score
26 Feb   Restaurant Restaurants  8383.0      4.3
22 Jun      Dineout Restaurants  7935.0      4.1
31 Mar   Restaurant Restaurants  7931.0      4.1
26 Jun       Amazon  E-commerce 22008.0      4.0
07 Feb       Amazon  E-commerce 21986.0      4.0


## Feature 8 - Spending Archetype Detection

A person can match multiple archetypes — we run every rule against the real numbers computed above and print whichever ones actually trigger, along with the metric that triggered them.

In [96]:
def pct_of_spend(categories):
  return spend_df[spend_df['category'].isin(categories)]['amount'].sum() / spend_total * 100

foodie_pct       = pct_of_spend(['Food Delivery', 'Restaurants', 'Cafe'])
qc_pct           = category_pct.get('Quick Commerce', 0)
ecommerce_pct    = category_pct.get('E-commerce', 0)
investments_pct  = category_pct.get('Investments', 0)
transport_pct    = category_pct.get('Transport', 0)
subscription_vendor_count = df.loc[df['category'] == 'Subscriptions', 'vendor_clean'].nunique()

def is_foodie():
  return foodie_pct > 25, foodie_pct

def is_quick_commerce_junkie():
  return qc_pct > 15, qc_pct

def is_shopaholic():
  return ecommerce_pct > 15, ecommerce_pct

def is_investor():
  return investments_pct > 15, investments_pct

def is_late_night_snacker():
  return late_night_food_pct > 50, late_night_food_pct

def is_cab_commuter():
  return transport_pct > 10, transport_pct

def is_subscription_lover():
  return subscription_vendor_count >= 5, subscription_vendor_count

def is_yolo_spender():
  return savings_rate < 10, savings_rate

def is_disciplined_saver():
  return savings_rate > 40, savings_rate

archetype_rules = [
    ("THE FOODIE",               is_foodie(),               "of spend on Food Delivery+Restaurants+Cafe"),
    ("THE QUICK COMMERCE JUNKIE", is_quick_commerce_junkie(), "of spend on Quick Commerce"),
    ("THE SHOPAHOLIC",           is_shopaholic(),            "of spend on E-commerce"),
    ("THE INVESTOR",             is_investor(),              "of spend on Investments"),
    ("THE LATE-NIGHT SNACKER",   is_late_night_snacker(),    "of Food Delivery orders after 9 PM"),
    ("THE CAB COMMUTER",         is_cab_commuter(),          "of spend on Transport"),
    ("THE SUBSCRIPTION LOVER",   is_subscription_lover(),    "distinct active subscription vendors"),
    ("THE YOLO SPENDER",         is_yolo_spender(),          "savings rate"),
    ("THE DISCIPLINED SAVER",    is_disciplined_saver(),     "savings rate"),
]

print("Archetype check results:\n")
detected_archetypes = []
for name, (matched, metric), unit in archetype_rules:
  flag = "-> MATCH" if matched else "   no match"
  print(f"{flag:<12} {name:<28} ({metric:.1f} {unit})")
  if matched:
    detected_archetypes.append((name, metric, unit))

Archetype check results:

   no match  THE FOODIE                   (17.3 of spend on Food Delivery+Restaurants+Cafe)
   no match  THE QUICK COMMERCE JUNKIE    (5.9 of spend on Quick Commerce)
-> MATCH     THE SHOPAHOLIC               (36.9 of spend on E-commerce)
-> MATCH     THE INVESTOR                 (15.4 of spend on Investments)
   no match  THE LATE-NIGHT SNACKER       (19.9 of Food Delivery orders after 9 PM)
   no match  THE CAB COMMUTER             (3.6 of spend on Transport)
   no match  THE SUBSCRIPTION LOVER       (4.0 distinct active subscription vendors)
-> MATCH     THE YOLO SPENDER             (-229.3 savings rate)
   no match  THE DISCIPLINED SAVER        (-229.3 savings rate)


## Invented archetype (THE SOLO SHOPAHOLIC)

In [97]:
top_vendor_name = vendor_totals.index[0]
top_vendor_pct = vendor_totals.iloc[0] / spend_total * 100
is_solo_shopaholic = top_vendor_pct > 15

print("THE SOLO SHOPAHOLIC (invented archetype)")
print(f"Rule: any single vendor > 15% of total spend.")
if is_solo_shopaholic:
    print(f"-> MATCH: {top_vendor_name} alone accounts for {top_vendor_pct:.1f}% of total spend.")
    detected_archetypes.append(("THE SOLO SHOPAHOLIC", top_vendor_pct, f"of spend on {top_vendor_name} alone"))
else:
    print(f"-> no match: top vendor ({top_vendor_name}) is only {top_vendor_pct:.1f}% of spend.")

THE SOLO SHOPAHOLIC (invented archetype)
Rule: any single vendor > 15% of total spend.
-> MATCH: Amazon alone accounts for 19.8% of total spend.


## Day-of-week analysis

In [98]:
spend_df['day_of_week'] = df.loc[spend_df.index, 'date'].dt.day_name()
dow_totals = spend_df.groupby('day_of_week')['amount'].sum()

weekend_total = dow_totals.get('Saturday', 0) + dow_totals.get('Sunday', 0)
weekday_total = dow_totals.sum() - weekend_total
weekend_avg_per_day = weekend_total / 2
weekday_avg_per_day = weekday_total / 5
weekend_premium_pct = (weekend_avg_per_day - weekday_avg_per_day) / weekday_avg_per_day * 100

print(f"Avg weekday spend/day : Rs. {weekday_avg_per_day:,.0f}")
print(f"Avg weekend spend/day : Rs. {weekend_avg_per_day:,.0f}")
print(f"Weekend premium       : {weekend_premium_pct:+.1f}%")
if weekend_premium_pct > 20:
    print("-> Weekends are meaningfully more expensive than weekdays for Rahul.")
else:
    print("-> Rahul's spending is fairly evenly spread across the week (no strong weekend spike).")

Avg weekday spend/day : Rs. 229,395
Avg weekend spend/day : Rs. 230,914
Weekend premium       : +0.7%
-> Rahul's spending is fairly evenly spread across the week (no strong weekend spike).


## Spend forecasting

In [99]:
last_three_months = month_pivot[['Apr', 'May', 'Jun']].values
forecast_next_month = last_three_months.mean(axis=1)

print(f"{'Category':<18} {'Forecast (Jul)':>16}")
print("-" * 36)
for cat, value in zip(month_pivot.index, forecast_next_month):
    print(f"{cat:<18} Rs. {value:>10,.0f}")

Category             Forecast (Jul)
------------------------------------
Cafe               Rs.      6,011
E-commerce         Rs.    100,093
Entertainment      Rs.      1,379
Food Delivery      Rs.     20,472
Fuel               Rs.     10,246
Groceries          Rs.      8,966
Investments        Rs.     42,028
Quick Commerce     Rs.     15,809
Rent               Rs.     18,000
Restaurants        Rs.     18,593
Subscriptions      Rs.      3,438
Transport          Rs.      9,807
Utilities          Rs.      6,797


## The Final Printed Report

In [100]:
def bar(pct, scale=2.5):
  return '#' * max(1, int(pct / scale))

W = 64
print("=" * W)
print("  SpendDNA REPORT - RAHUL SHARMA".center(W))
print(f"  6 months - {len(df):,} transactions - Jan to Jun 2024".center(W))
print("=" * W)

print()
print("  EXECUTIVE SUMMARY")
print(f"    Total credits           : Rs. {total_credits:,.0f}")
print(f"    Total debits            : Rs. {total_debits:,.0f}")
overspend_tag = "(overspending)" if net_change < 0 else "(saving)"
print(f"    Net change              : Rs. {net_change:,.0f}        {overspend_tag}")
savings_tag = "(BURNING SAVINGS)" if savings_rate < 0 else ("(healthy)" if savings_rate > 20 else "(tight)")
print(f"    Savings rate            : {savings_rate:.1f}%               {savings_tag}")
print(f"    Transactions            : {len(df):,}")
print(f"    Unique vendors          : {df['vendor_clean'].nunique()}")

print()
print("  TOP CATEGORIES (% of spend total)")
for cat, pct in category_pct.head(5).items():
  amt = category_totals[cat]
  print(f"    {cat:<15} {bar(pct):<26} {pct:>5.1f}%     Rs. {amt:>10,.0f}")

print()
print("  TOP VENDORS")
for vendor, amt in vendor_totals.head(5).items():
  count = vendor_counts[vendor]
  print(f"    {vendor:<20} Rs. {amt:>10,.0f}   ({count} txns)")

print()
print("  TIME-OF-DAY PATTERNS")
print(f"    Food Delivery late-night (9PM-2AM) : {late_night_food_pct:.1f}% of orders")
print(f"    Cafe morning runs (8-11AM)         : {morning_cafe_pct:.1f}% of spend")

print()
print(f"  MONTHLY TREND ({category_pct.index[0]})")
top_cat_row = month_pivot.loc[category_pct.index[0]]
for month_name, amt in top_cat_row.items():
  print(f"    {month_name} Rs. {amt:>8,.0f} {bar(amt, scale=5000)}")

print()
print("  TOP ANOMALIES (2+ stddev from category mean)")
for _, row in anomalies.head(5).iterrows():
  date_str = row['date'].strftime('%d %b')
  print(f"    {date_str} - {row['vendor_clean']:<15} Rs. {row['amount']:>8,.0f}   (z={row['z_score']:.1f})")

print()
print("  RAHUL'S SPENDING ARCHETYPES")
if detected_archetypes:
  for name, metric, unit in detected_archetypes:
    print(f"    -> {name:<26} ({metric:.1f} {unit})")
else:
  print("    No archetype thresholds were crossed.")

print()
print("=" * W)

                  SpendDNA REPORT - RAHUL SHARMA                
         6 months - 1,310 transactions - Jan to Jun 2024        

  EXECUTIVE SUMMARY
    Total credits           : Rs. 509,774
    Total debits            : Rs. 1,678,901
    Net change              : Rs. -1,169,127        (overspending)
    Savings rate            : -229.3%               (BURNING SAVINGS)
    Transactions            : 1,310
    Unique vendors          : 41

  TOP CATEGORIES (% of spend total)
    E-commerce      ##############              36.9%     Rs.    593,959
    Investments     ######                      15.4%     Rs.    248,160
    Restaurants     ###                          7.9%     Rs.    127,290
    Food Delivery   ##                           7.4%     Rs.    119,501
    Rent            ##                           6.7%     Rs.    108,000

  TOP VENDORS
    Amazon               Rs.    318,612   (77 txns)
    Zerodha              Rs.    210,000   (14 txns)
    Flipkart             Rs.    177,

## Key Insights

1. **E-commerce, not food delivery, is Rahul's biggest leak.** E-commerce is ~37% of his spend and Amazon alone is close to 20% of it — well above any other single vendor — driven by a handful of ₹15,000–22,000 marketplace orders that also show up as the largest anomalies in Feature 7.
2. **He's spending roughly double what he earns.** A savings rate deep in negative triple digits means his balance is being drawn down every month, not just in the months with a big one-off purchase — the monthly trend table shows E-commerce and Investments as consistently the two largest line items in every single month from Jan to Jun.
3. **Investments are the one bright spot.** ~15% of spend goes to Zerodha SIPs/coin purchases, on a steady ₹15,000/month rhythm — the discipline is there, it's just being outpaced by discretionary online shopping running at more than double the invested amount.

## Reflection

**What was hardest:** The vendor mapping part. `df['Description'].unique()` had close to 280 different raw strings for the same handful of vendors — different UPI handles, POS prefixes, and even parent company names instead of the brand name (like BUNDL Technologies for Swiggy). Building a keyword dictionary that correctly grouped all of these into one canonical vendor name, without missing any or mixing two vendors together, took the most time and testing.

**What surprised me:** How much Rahul spends on E-commerce and Investments compared to everything else — and that he isn't saving anything for the future. His spending is actually more than what he earns, which means he's going deeper into debt every month rather than building any savings. Seeing that laid out in numbers (a savings rate deep in the negatives) made it feel a lot more real than just reading "he overspends" would have.

**What I'd build next:** A way to flag this kind of pattern earlier — something that warns when monthly spend is consistently outpacing income, instead of only seeing it after totaling six months of transactions.

In [101]:
# AI-assisted: z-score anomaly logic and final report formatting drafted with AI;
# I adjusted the report's bar-chart scaling after testing showed E-commerce bars
# were too long to read.